<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Mini_projet_W6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Resources
Content: []
Mini Project: Sentiment Assistant with BERT Fine-Tuning


Welcome to the final day mini project!

In this lab you will fine-tune bert-base-uncased on movie reviews, evaluate the model, and connect the dots with real customer-support scenarios. Every section explains why we take a step, what to look for, and where an illustration could make the concept stick for your learners.

Prerequisites & Story Setup
Scenario: The support analytics team wants a dependable sentiment signal for long-form feedback so they can escalate angry customers before churn happens.
Environment needs: Python 3.9+, a GPU-enabled runtime (Colab, Kaggle, or a local GPU machine), and ~6 GB of free VRAM. CPU-only will work but training takes longer.
Packages: tensorflow, tensorflow-datasets, transformers, accelerate, and evaluate, all of which appeared earlier in the course, so you already used these tools before.


# Run once in a fresh environment

In [ ]:
pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

🔍 Here you will notice that we reuse the exact toolchain from Day 3 and 4. This reinforces continuity and lets us focus on the new workflow rather than new libraries.

Here’s a quick illustration idea to set the stage of the methodology we will explore:

alt text



Imports & Hardware Check
We always begin by confirming versions and hardware. If a learner sees GPU devices: [], they immediately know to switch the runtime (when we are using Google colab, no further installations and configurations are needed).

In [ ]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

alt text



Load the IMDB Reviews Dataset
We rely on IMDB because it is balanced (25k positive / 25k negative) and already split. Learners should recognize it from earlier sentiment demos.

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)

💬 Here you will notice TFDS returns both the dataset objects and metadata. Point out that as_supervised=True yields (text, label) pairs, exactly what our model expects.

Take a quick peek at samples to make sentiment concrete:

In [ ]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

Tokenizer Setup & Data Pipeline
BERT uses WordPiece tokenization to handle rare or unknown words by breaking them into subword units, ensuring coverage and efficient vocabulary size. It adds [CLS] and [SEP] tokens to mark sentence boundaries and enable tasks like classification and sentence-pair modeling. Attention masks tell the model which tokens are real versus padding, ensuring attention is only computed over valid inputs.

In [ ]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

🧠 we import this tokenizer to reuse the same vocabulary the base model learned in 2018

Next, convert raw bytes → token IDs, attention masks, and segment IDs.

In [ ]:
def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    return {
        "input_ids": encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label

def prepare_dataset(dataset):
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

🗒️ Remember that tf.py_function lets us keep Hugging Face tokenization logic inside a TF pipeline, so we don’t manually juggle NumPy arrays. Also shuffling and prefetching stabilize training throughput.



Initialize the Fine-Tuning Model
Now we load TFBertForSequenceClassification, which already bundles the encoder + classification head.

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

💡 we import this model to reuse the 110M parameters learned on BooksCorpus + Wikipedia. We only fine-tune for a couple of epochs, which is why learning rates in the 2e-5 range are standard.



Train and Monitor
On a T4 GPU (the one you find in Google Colab), two epochs take ~15 minutes. Please watch both training and validation accuracy.

In [ ]:
EPOCHS = 2  # increase to 3 if time allows
history = model.fit(
    x=train_ds,
    epochs=EPOCHS,
    validation_data=test_ds
)

📈 Point out how the validation accuracy plateau indicates when to stop. Also encourage screenshots of the learning curves for portfolios.



Evaluate on the Held-Out Test Set
Even though model.fit already reports validation metrics, we re-run evaluation on the untouched test pipeline to mimic production QA.

In [ ]:
eval_metrics = model.evaluate(test_ds)
print("Evaluation metrics:", eval_metrics)

✅ *Here you will notice whether accuracy crosses the ~0.90 classroom benchmark.

Use this to discuss acceptable error rates for real support teams.*

Build a Reusable Inference Helper
Wrap everything in a function so that you can paste real support transcripts and get an instant score.

In [ ]:
def predict_sentiment(text: str):
    encoded_input = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf" # Return TensorFlow tensors
    )

    outputs = model(encoded_input)
    logits = outputs.logits
    probs = tf.nn.softmax(logits)[0] # Get probabilities for the first (and only) item in batch
    label = "Positive" if tf.argmax(probs).numpy() == 1 else "Negative"
    return label, float(probs.numpy().max())

custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

For the previous example, you should see Prediction: Positive (confidence=0.5) slightly more or less.

🧭 confidence scores are essential when deciding whether to auto-respond or escalate to a human.



Reflection & Next Steps
Why fine-tuning matters: You reused a public checkpoint to hit >90% accuracy with minimal data.
Transferable skills: Everything here also applies to classification tasks in HR, legal, or product analytics.
What you can do with this: domain adaptation (collect your company’s emails), multilingual checkpoints (DistilBERT multilingual, XLM-R), and monitoring (log data drift, build dashboards).

## Reflection Questions Answers:

**1. What lever (data cleaning, hyperparameters, more epochs) most improved results?**

For fine-tuning pre-trained models like BERT on a well-established dataset like IMDB reviews, the most impactful levers are usually:
*   **More epochs (up to a point):** Fine-tuning for a few epochs (e.g., 2-4) often significantly improves performance by allowing the model to adapt to the specific task's data distribution. However, too many epochs can lead to overfitting.
*   **Appropriate learning rate:** A small learning rate (like 2e-5) is crucial to avoid corrupting the pre-trained weights too quickly.
*   **Dataset quality and size:** While this project uses a standard dataset, in real-world scenarios, having a large, clean, and relevant labeled dataset for fine-tuning is paramount.

Less impactful, but still relevant:
*   **Data cleaning:** The IMDB dataset is relatively clean. For raw, noisy data, significant cleaning would be a primary lever.
*   **Hyperparameters (batch size, etc.):** These can offer marginal gains after the primary levers are addressed.

**2. Where would you add guardrails before deploying this sentiment signal live?**

*   **Confidence Thresholds:** Implement a confidence threshold. If the model's confidence score is below a certain value (e.g., 0.7 or 0.8), flag the review for human review rather than making an automatic decision. This prevents misclassifications in ambiguous cases.
*   **Human-in-the-Loop Feedback:** Establish a process for human agents to correct misclassifications. This feedback loop can be used to periodically retrain or further fine-tune the model, improving its performance over time on real-world data.
*   **Bias Detection and Mitigation:** Regularly audit the model for biases against certain demographics, keywords, or topics. This could involve checking performance on subsets of data or using interpretability tools.
*   **Out-of-Domain Detection:** The model is trained on movie reviews. If deployed to a customer support context, it might perform poorly on highly specific product complaints or technical queries. Implement mechanisms to detect and flag out-of-domain text for human review.
*   **Data Drift Monitoring:** Monitor the characteristics of incoming customer support text. If the language, topics, or sentiment distribution start to drift significantly from the training data, it might indicate that the model needs retraining or recalibration.
*   **Edge Case Handling:** Explicitly define how to handle reviews that are short, sarcastic, contain multiple sentiments, or are multilingual.
*   **Explainability:** Provide some level of explainability for the model's predictions (e.g., highlighting key words that led to a positive or negative classification) to aid human agents in their review.

**3. Which stakeholders benefit the most (support lead, product manager, compliance officer)?**

*   **Support Lead / Customer Service Manager:** Benefits immensely by:
    *   **Proactive Escalation:** Identifying angry or at-risk customers quickly, allowing for immediate human intervention and churn prevention.
    *   **Resource Allocation:** Prioritizing support tickets based on sentiment severity.
    *   **Agent Performance:** Providing insights into types of interactions and agent effectiveness.
    *   **Training:** Identifying common pain points or challenging customer interactions for agent training.
*   **Product Manager:** Benefits by:
    *   **Feature Prioritization:** Gaining insights into frequently requested features or common complaints, informing product roadmap decisions.
    *   **Bug Detection:** Quickly identifying widespread issues or bugs reported by customers through negative sentiment.
    *   **User Experience (UX) Improvement:** Understanding friction points in the user journey.
*   **Compliance Officer (potentially):** Could benefit by:
    *   **Risk Identification:** Flagging conversations that might involve legal or compliance risks (e.g., hate speech, inappropriate content, or regulatory violations). While direct sentiment analysis might not be sufficient, it can be a component of a larger system.
    *   **Audit Trails:** Providing a record of sentiment analysis for specific customer interactions, which could be useful in disputes.

All three roles gain valuable insights, but the **Support Lead** likely sees the most direct and immediate benefit in improving day-to-day operations and customer retention.